# mAP scatters (complex + gene KO) + GSEA enrichment

Two scatter panels and a GSEA bar plot comparing the cell-dino phase-only OPS embedding against the sVAEplus crop-seq embedding.

1. **Complex mAP scatter** — per-complex `mean_average_precision` from crop-seq vs cell-dino; top-K most-divergent complexes labelled by complex name.
2. **Gene-KO mAP scatter** — per-perturbation mAP, crop-seq vs cell-dino; top-K labelled by gene symbol.
3. **GSEA bar plot** — top imaging- and RNA-favoured GO BP / CC terms from a prerank GSEA on (cell-dino − crop-seq) per-gene mAP, FDR-shaded.

Inputs live under `../../data/figures/figure_5/`. Panels are saved as `.pdf` + `.svg` into `../../output/figure_5/`.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from adjustText import adjust_text


# ---- Paper-wide plotting parameters (inlined from former parameters.py) ----
class P:
    # Output / sizing
    DPI                  = 300
    SAVEFIG_FORMAT       = 'svg'
    PANEL_SIZE_IN        = (2.4, 2.0)
    SQUARE_PANEL_SIZE_IN = (2.4, 2.4)

    # Fonts (panels are small so default sizes are small too)
    FONT_FAMILY           = 'Arial'
    FONT_SIZE_AXIS_LABEL  = 8
    FONT_SIZE_TICK        = 6
    FONT_SIZE_TITLE       = 10
    FONT_SIZE_LEGEND      = 8
    FONT_SIZE_PANEL_LABEL = 14

    # Colours
    COLOR_BACKGROUND  = '#bdbdbd'   # grey, used for null / "all" distributions
    COLOR_HIGHLIGHT   = '#e07a3a'   # orange, RNA-side
    COLOR_HIGHLIGHT_2 = '#1f8a8a'   # teal, image-side
    COLOR_TEXT        = '#222222'
    COLOR_AXIS        = '#222222'

    # Lines
    LINEWIDTH      = 1.4
    KDE_LINEWIDTH  = 1.8
    LINESTYLE_RNA  = '--'
    LINESTYLE_IMG  = '-'

    @staticmethod
    def apply_style():
        plt.rcParams.update({
            'font.family':        P.FONT_FAMILY,
            'font.size':          P.FONT_SIZE_AXIS_LABEL,
            'axes.labelsize':     P.FONT_SIZE_AXIS_LABEL,
            'axes.titlesize':     P.FONT_SIZE_TITLE,
            'xtick.labelsize':    P.FONT_SIZE_TICK,
            'ytick.labelsize':    P.FONT_SIZE_TICK,
            'legend.fontsize':    P.FONT_SIZE_LEGEND,
            'axes.linewidth':     P.LINEWIDTH,
            'xtick.major.width':  P.LINEWIDTH,
            'ytick.major.width':  P.LINEWIDTH,
            'text.color':         P.COLOR_TEXT,
            'axes.edgecolor':     P.COLOR_AXIS,
            'axes.labelcolor':    P.COLOR_TEXT,
            'xtick.color':        P.COLOR_AXIS,
            'ytick.color':        P.COLOR_AXIS,
            'svg.fonttype':       'none',  # keep text editable in Illustrator
            'pdf.fonttype':       42,
            'ps.fonttype':        42,
        })

P.apply_style()


FIGURE_DATA = Path("../../data/figures/figure_5")
FIGURES_DIR = Path("../../output/figure_5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CROPSEQ_MAP          = FIGURE_DATA / "cropseq_ebi_map.csv"
IMAGE_MAP            = FIGURE_DATA / "celldino_phase_only_ebi.csv"
IMAGE_MAP_PER_GENE   = FIGURE_DATA / "celldino_phase_only_distinctiveness.csv"
CROPSEQ_MAP_PER_GENE = FIGURE_DATA / "svaeplus_distinctiveness_std_ntc.csv"
COMPLEX_YAML         = FIGURE_DATA / "EBI_complexes_v1_updated_gene_names.yaml"

TOP_K = 8
POINT_COLOR = P.COLOR_HIGHLIGHT_2          # teal, neutral between the two modalities
DIAG_COLOR  = '#888888'
LABEL_FONT_SIZE = 5

_COMPLEX_RE = re.compile(r'\bcomplete\s+complex\b|\bcomplex\b', re.IGNORECASE)
def clean_complex_name(name):
    """Strip the redundant 'complex' suffix and tidy whitespace/punctuation."""
    return re.sub(r'\s+', ' ', _COMPLEX_RE.sub('', name)).strip(' ,-')


def save_panel(fig, stem):
    """Save fig as both .pdf and .svg into FIGURES_DIR."""
    for ext in ('pdf', 'svg'):
        out = FIGURES_DIR / f'{stem}.{ext}'
        fig.savefig(out, dpi=P.DPI, bbox_inches='tight')
        print(f'  wrote {out}')

In [ ]:
# ---- mAP table (crop-seq vs image / cell_dino) — joined on complex_num,
#      decorated with complex name + member count from the EBI YAML ----
with open(COMPLEX_YAML) as f:
    chad_yaml = yaml.safe_load(f)
num_to_name    = {int(k): v['name'] for k, v in chad_yaml.items()}
num_to_members = {int(k): len(v['genes']) for k, v in chad_yaml.items()}

cs_map = pd.read_csv(CROPSEQ_MAP)
im_map = pd.read_csv(IMAGE_MAP)

shared = set(cs_map['complex_num']) & set(im_map['complex_num'])
cs_sub = (cs_map[cs_map['complex_num'].isin(shared)]
            .set_index('complex_num')[['mean_average_precision']]
            .rename(columns={'mean_average_precision': 'crop_seq'}))
im_sub = (im_map[im_map['complex_num'].isin(shared)]
            .set_index('complex_num')[['mean_average_precision']]
            .rename(columns={'mean_average_precision': 'cell_dino'}))
map_df = cs_sub.join(im_sub)
map_df['name']      = map_df.index.map(num_to_name)
map_df['n_members'] = map_df.index.map(num_to_members)
print(f'shared complexes in mAP scatter: {len(map_df)}')

In [ ]:
# ---- Complex mAP scatter (points sized by member count) ----
def members_to_size(n_members, s_min=8, s_max=80):
    vmin, vmax = n_members.min(), n_members.max()
    if vmin == vmax:
        return pd.Series(s_min, index=n_members.index)
    return s_min + (n_members - vmin) / (vmax - vmin) * (s_max - s_min)

sizes = members_to_size(map_df['n_members'])

fig, ax = plt.subplots(figsize=P.SQUARE_PANEL_SIZE_IN, dpi=P.DPI)
ax.scatter(map_df['crop_seq'], map_df['cell_dino'], s=sizes,
           color=POINT_COLOR, alpha=0.65, linewidths=0)
ax.plot([0, 1], [0, 1], linestyle='--', color=DIAG_COLOR, linewidth=0.8, alpha=0.7)

map_df = map_df.assign(diff=(map_df['crop_seq'] - map_df['cell_dino']).abs())
top = map_df.nlargest(TOP_K, 'diff').reset_index()

# Extra headroom above/below the data so labels can stagger across 2 rows.
ax.set_xlim(-0.02, 1.05); ax.set_ylim(-0.02, 1.05)
ax.set_xticks([0, 0.5, 1.0]); ax.set_yticks([0, 0.5, 1.0])
ax.set_xlabel('crop-seq mAP')
ax.set_ylabel('cell-dino mAP')
ax.set_title(f'Protein Complex mAP (n={len(map_df)})', fontsize=P.FONT_SIZE_TITLE)
ax.set_aspect('equal')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Within each strip, sort labels by source x and stagger across 2 sub-rows so
# adjacent labels never share a y — their horizontal bounding boxes can bleed
# past each other without visually colliding.
def place_strip(rows, y_base, direction, row_step=0.09):
    if rows.empty:
        return
    rows = rows.sort_values('crop_seq').reset_index(drop=True)
    n = len(rows)
    label_xs = np.linspace(0.0, 1.05, n)
    for i, (x_lbl, (_, row)) in enumerate(zip(label_xs, rows.iterrows())):
        sub_row = i % 2                          # 0, 1, 0, 1, ...
        y_lbl = y_base + direction * sub_row * row_step
        # ax.annotate(
        #     clean_complex_name(row['name']),
        #     xy=(row['crop_seq'], row['cell_dino']),
        #     xytext=(x_lbl, y_lbl),
        #     fontsize=LABEL_FONT_SIZE, color='black',
        #     ha='center', va='center',
        #     arrowprops=dict(arrowstyle='-', color='black', lw=0.4,
        #                     shrinkA=0, shrinkB=2),
        # )

above = top[top['cell_dino'] >  top['crop_seq']]
below = top[top['cell_dino'] <= top['crop_seq']]
place_strip(above, y_base=1.12, direction=+1)
place_strip(below, y_base=-0.12, direction=-1)

fig.tight_layout()
save_panel(fig, 'mAP_complex_scatter')
plt.show()

In [ ]:
# ---- Gene-KO mAP scatter (crop-seq vs image) ----
cs_gene = (pd.read_csv(CROPSEQ_MAP_PER_GENE)[['perturbation', 'mean_average_precision']]
             .rename(columns={'mean_average_precision': 'crop_seq'}))
im_gene = (pd.read_csv(IMAGE_MAP_PER_GENE)[['perturbation', 'mean_average_precision']]
             .rename(columns={'mean_average_precision': 'cell_dino'}))
gene_df = (cs_gene.merge(im_gene, on='perturbation', how='inner')
                  .loc[lambda d: ~d['perturbation'].str.startswith('NTC')]
                  .reset_index(drop=True))
print(f'shared genes in per-gene mAP scatter: {len(gene_df)}')

fig, ax = plt.subplots(figsize=P.SQUARE_PANEL_SIZE_IN, dpi=P.DPI)
ax.scatter(gene_df['crop_seq'], gene_df['cell_dino'],
           s=6, color=POINT_COLOR, alpha=0.45, linewidths=0)
ax.plot([0, 1], [0, 1], linestyle='--', color=DIAG_COLOR, linewidth=0.8, alpha=0.7)

gene_df = gene_df.assign(diff=(gene_df['crop_seq'] - gene_df['cell_dino']).abs())
top = gene_df.nlargest(TOP_K, 'diff').reset_index(drop=True)
# texts = [
#     ax.text(row['crop_seq'], row['cell_dino'], row['perturbation'],
#             fontsize=LABEL_FONT_SIZE, color='black')
#     for _, row in top.iterrows()
# ]

ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.set_xticks([0, 0.5, 1.0]); ax.set_yticks([0, 0.5, 1.0])
ax.set_xlabel('crop-seq mAP')
ax.set_ylabel('cell-dino mAP')
ax.set_title(f'Gene KO mAP (n={len(gene_df)})', fontsize=P.FONT_SIZE_TITLE)
ax.set_aspect('equal')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# adjust_text(
#     texts, ax=ax,
#     arrowprops=dict(arrowstyle='-', color='black', lw=0.4, shrinkA=2, shrinkB=2),
#     expand=(1.6, 1.6),
#     force_text=(0.6, 0.6),
#     force_static=(0.4, 0.4),
#     force_pull=(0.02, 0.02),
#     max_move=(40, 40),
# )

fig.tight_layout()
save_panel(fig, 'mAP_KO_scatter')
plt.show()

In [ ]:
# ---- GSEA on per-gene mAP difference (image - crop-seq) ----
import gseapy as gp

# Positive score = imaging-favoured; negative = RNA-favoured.
rnk = (gene_df.assign(score=gene_df['cell_dino'] - gene_df['crop_seq'])
              [['perturbation', 'score']]
              .sort_values('score', ascending=False)
              .reset_index(drop=True))
print(f'ranked genes: {len(rnk)}  range: [{rnk["score"].min():.3f}, {rnk["score"].max():.3f}]')

LIBRARIES = {
    'GO_BP_2025': str(FIGURE_DATA / 'GO_Biological_Process_2025.gmt'),
    'GO_CC_2025': str(FIGURE_DATA / 'GO_Cellular_Component_2025.gmt'),
}

gsea_results = {}
for name, gmt in LIBRARIES.items():
    out = FIGURES_DIR / f'gsea_mAP_diff_{name}'
    pre = gp.prerank(rnk=rnk, gene_sets=gmt, outdir=str(out),
                     min_size=5, max_size=500, permutation_num=1000,
                     seed=0, threads=4, verbose=False)
    res = pre.res2d.sort_values('NES', key=abs, ascending=False).reset_index(drop=True)
    gsea_results[name] = res
    print(f'\n=== {name}  (n_terms={len(res)}) ===')
    print(res[['Term', 'NES', 'NOM p-val', 'FDR q-val', 'Tag %']].head(10).to_string(index=False))
    print('  --- top imaging-favoured ---')
    print(res.sort_values('NES', ascending=False)[['Term','NES','FDR q-val']].head(5).to_string(index=False))
    print('  --- top RNA-favoured ---')
    print(res.sort_values('NES', ascending=True )[['Term','NES','FDR q-val']].head(5).to_string(index=False))

In [ ]:
# ---- GSEA bar plot: top imaging vs RNA terms per library, coloured by FDR q ----
TOP_PER_DIRECTION = 8
Q_EPS = 1e-4                       # floor so q==0 stays on the colour scale
CMAP = plt.get_cmap('magma_r')     # darker = more significant

def _shorten(term, n=55):
    t = re.sub(r'\s*\(GO:\d+\)\s*$', '', term)
    return t if len(t) <= n else t[: n - 1] + '…'

def _top_by_sign(res, k):
    """Top-k positive- and negative-NES rows, disjoint by sign."""
    pos = res[res['NES'] > 0].sort_values('NES', ascending=False).head(k)
    neg = res[res['NES'] < 0].sort_values('NES', ascending=True ).head(k)
    return pd.concat([neg, pos], ignore_index=True).sort_values('NES').reset_index(drop=True)

# Shared colour scale across both panels so bars are visually comparable.
all_q = pd.concat([
    pd.to_numeric(res['FDR q-val'], errors='coerce')
    for res in gsea_results.values()
]).dropna()
vmax = float(-np.log10(max(all_q.min(), Q_EPS)))
vmin = 0.0                          # q == 1 maps to 0 on −log10 scale
norm = plt.Normalize(vmin=vmin, vmax=vmax)

fig, axes = plt.subplots(
    1, 2,
    figsize=(2 * P.PANEL_SIZE_IN[0] + 1.2, P.PANEL_SIZE_IN[1] + 1.8),
    dpi=P.DPI,
)

for i, (ax, (lib_name, res)) in enumerate(zip(axes, gsea_results.items())):
    res = res.copy()
    res['NES'] = pd.to_numeric(res['NES'], errors='coerce')
    res['FDR q-val'] = pd.to_numeric(res['FDR q-val'], errors='coerce')
    res = res.dropna(subset=['NES', 'FDR q-val'])
    plot_df = _top_by_sign(res, TOP_PER_DIRECTION)
    labels = [_shorten(t) for t in plot_df['Term']]
    nlog_q = -np.log10(plot_df['FDR q-val'].clip(lower=Q_EPS))
    colors = CMAP(norm(nlog_q))
    y = np.arange(len(plot_df))
    ax.barh(y, plot_df['NES'], color=colors, linewidth=0)
    ax.axvline(0, color='#555555', linewidth=0.6)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=6)
    ax.set_xlabel('NES  (← RNA-favoured  |  imaging-favoured →)')
    ax.set_title(lib_name, fontsize=P.FONT_SIZE_TITLE)
    ax.spines['top'].set_visible(False)
    if i == 1:                              # right plot: move tick labels to the right
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position('right')
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_linewidth(0.6)
    else:
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.6)

# Shared colour bar for −log10(FDR q)
sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, orientation='horizontal',
                    fraction=0.04, pad=0.18, aspect=40)
cbar.set_label('−log10(FDR q-val)', fontsize=P.FONT_SIZE_LEGEND)
cbar.ax.tick_params(labelsize=P.FONT_SIZE_TICK)

save_panel(fig, 'gsea_top_terms')
plt.show()